In [1]:
# version 8 2025.12.05 - PyCaret v7 기반 + AutoGluon + fastText

import torch
print(torch.cuda.is_available())  # True면 정상
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # GPU 이름 출력

# 학원 Workstation 환경은 32GB Ram에, GPU는 NVIDIA Geforce GTX 1660 Super,
# CPU는 AMD RYZEN 5 Pro 4650 with Radeon Graphics

True
NVIDIA GeForce GTX 1660 SUPER


In [ ]:
# version 8 2025.12.05 - PyCaret v7 기반 + AutoGluon + fastText

import torch
print(torch.cuda.is_available())  # True면 정상
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))  # GPU 이름 출력

# 학원 Workstation 환경은 32GB Ram에, GPU는 NVIDIA Geforce GTX 1660 Super,
# CPU는 AMD RYZEN 5 Pro 4650 with Radeon Graphics

import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
import re
import warnings
from tqdm import tqdm
from gensim.models import FastText
from sentence_transformers import SentenceTransformer
from pycaret.regression import save_model, load_model

warnings.filterwarnings("ignore")

from pycaret.regression import *
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 🔹 AutoGluon
from autogluon.tabular import TabularPredictor


class MercariPyCaretAnalyzer7:
    """
    Mercari Price Suggestion Challenge를 위한 PyCaret + AutoGluon 기반 머신러닝 파이프라인

    주요 기능: 데이터 전처리, 텍스트 벡터화(TF-IDF/FastText/BERT),
              피처 엔지니어링, PyCaret 기반 모델 학습/블렌딩,
              AutoGluon Tabular 앙상블, 성능 평가/서브미션 생성

    Parameters
    ----------
    data_dir : str, default="../data"
    images_dir : str, default="../images"
    results_dir : str, default="../results"
    model_dir : str, default="../models"
    """

    # __init__ ##############################
    def __init__(self, data_dir="../data", images_dir="../images",
                 results_dir="../results", model_dir="../models"):
        self.data_dir     = data_dir
        self.images_dir   = images_dir
        self.results_dir  = results_dir
        self.model_dir    = model_dir

        self.train        = None
        self.test         = None

        # PyCaret 쪽
        self.best_model   = None
        self.setup_result = None
        self.models       = {}   # 여러 모델을 메모리에 보관

        # AutoGluon 쪽
        self.ag_predictor = None

        self.metrics = {}

        os.makedirs(self.images_dir,  exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)
        os.makedirs(self.model_dir,   exist_ok=True)
    # eof -----------------------------------

    # _collapse_rare_values #################
    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        """희귀값 통합 (상위 top_k만 유지, 나머지는 rare_label로 묶기)"""
        combined = pd.concat([self.train[col], self.test[col]], axis=0)
        value_counts = combined.value_counts()
        top_values = set(value_counts.index[:top_k])
        self.train[col] = self.train[col].apply(lambda x: x if x in top_values else rare_label)
        self.test[col]  = self.test[col].apply(lambda x: x if x in top_values else rare_label)
        del combined, value_counts
        gc.collect()
    # eof -----------------------------------

    # _simple_normalize #####################
    def _simple_normalize(self, text: str) -> str:
        """
        Mercari 상위 솔루션 스타일의 간단 텍스트 정규화:
        - 소문자
        - _, -, ., / → 공백
        - 숫자 → 'num'
        - 다중 공백 정리
        """
        text = str(text).lower()
        text = re.sub(r"[_\-\./]", " ", text)
        text = re.sub(r"\d+", " num ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    # eof -----------------------------------

    # _clean_item_description ###############
    def _clean_item_description(self, text: str) -> str:
        """
        description 특이값 정리:
        - 'No description yet', 'no description', 'N/A' 등은 'no description'으로 통합
        - 상위권 솔루션들이 자주 하던 패턴을 단순화한 버전
        """
        text = str(text)
        lower = text.lower()
        if lower in ["no description yet", "no description", "n/a", "na"]:
            return "no description"
        return text
    # eof -----------------------------------

    # _stratified_sample ####################
    def _stratified_sample(self, frac=0.35, bins=10):
        """price 분포를 유지한 층화 샘플링 (언더샘플링)"""
        self.train["price_bin"] = pd.qcut(self.train["price"], q=bins, duplicates="drop")
        sampled = self.train.groupby("price_bin", group_keys=False).apply(
            lambda x: x.sample(frac=frac, random_state=23)
        )
        self.train = sampled.drop(columns=["price_bin"]).reset_index(drop=True)
        gc.collect()
        print(f"⚠️ Stratified undersampling 적용: train {self.train.shape}")
    # eof -----------------------------------

    # load_data #############################
    def load_data(self, train_file="train.tsv", test_file="test.tsv",
                  sep="\t", undersample_frac=0.35):
        """데이터 로딩 및 전처리"""
        print("📂 데이터 로딩 시작...")
        self.train = pd.read_csv(os.path.join(self.data_dir, train_file), sep=sep)
        self.test  = pd.read_csv(os.path.join(self.data_dir, test_file),  sep=sep)

        print(f"원본: train {self.train.shape}, test {self.test.shape}")

        # price > 0 + NaN 제거, log1p 변환
        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        self.train["price"] = np.log1p(self.train["price"])

        # 언더샘플링으로 메모리/시간 조절
        if undersample_frac:
            self._stratified_sample(frac=undersample_frac)

        # 기본 텍스트/카테고리 처리
        for df in [self.train, self.test]:
            # category_name 분해
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (x.split("/") if isinstance(x, str) and "/" in x
                               else ["missing"] * 3)
                )
            )

            df["brand_name"]       = df["brand_name"].fillna("Unknown").astype(str)
            df["item_description"] = df["item_description"].fillna("No description").astype(str)
            df["name"]             = df["name"].fillna("No name").astype(str)

            # description 클린 버전 적용
            df["item_description"] = df["item_description"].apply(self._clean_item_description)

            # category_name은 분리했으므로 삭제
            df.drop(columns=["category_name"], inplace=True)

        print("🔄 희귀값 통합 중...")
        self._collapse_rare_values("brand_name", 5000, "Other_brand")
        self._collapse_rare_values("main_cat",  1000, "Other_main")
        self._collapse_rare_values("sub_cat",   1000, "Other_sub")
        self._collapse_rare_values("sub_sub_cat", 1000, "Other_sub_sub")

        print("📏 피처 생성 중...")
        for df in [self.train, self.test]:
            df["name_len_char"] = df["name"].str.len()
            df["name_len_word"] = df["name"].str.split().str.len()
            df["desc_len_char"] = df["item_description"].str.len()
            df["desc_len_word"] = df["item_description"].str.split().str.len()

            df["has_brand_in_name"] = df.apply(
                lambda r: 1 if r["brand_name"].lower() in r["name"].lower() else 0,
                axis=1,
            )
            df["has_brand_in_desc"] = df.apply(
                lambda r: 1 if r["brand_name"].lower() in r["item_description"].lower() else 0,
                axis=1,
            )

            df["shipping"]          = df["shipping"].astype("category")
            df["item_condition_id"] = df["item_condition_id"].astype("category")

        gc.collect()
        print(f"✅ 데이터 로드 완료: train {self.train.shape}, test {self.test.shape}")
    # eof -----------------------------------

    # vectorize_text_tfidf ##################
    def vectorize_text_tfidf(self, max_features_name=15000,
                             max_features_desc=20000, n_components=150):
        """TF-IDF 벡터화 (참고용, 현재 주력은 fastText)"""
        print("🔍 TF-IDF 벡터화 시작...")

        # name
        vec_name = TfidfVectorizer(
            max_features=max_features_name, ngram_range=(1, 2),
            min_df=3, max_df=0.95, sublinear_tf=True, dtype=np.float32
        )
        name_train = vec_name.fit_transform(self.train["name"])
        name_test  = vec_name.transform(self.test["name"])

        svd_name = TruncatedSVD(
            n_components=min(n_components, name_train.shape[1] - 1),
            random_state=23,
        )
        name_train_svd = svd_name.fit_transform(name_train)
        name_test_svd  = svd_name.transform(name_test)
        print(f"   - name SVD: {name_train_svd.shape}, 설명력={svd_name.explained_variance_ratio_.sum():.2%}")

        del name_train, name_test, vec_name
        gc.collect()

        # desc
        vec_desc = TfidfVectorizer(
            max_features=max_features_desc, ngram_range=(1, 2),
            min_df=3, max_df=0.95, sublinear_tf=True, dtype=np.float32
        )
        desc_train = vec_desc.fit_transform(self.train["item_description"])
        desc_test  = vec_desc.transform(self.test["item_description"])

        svd_desc = TruncatedSVD(
            n_components=min(n_components, desc_train.shape[1] - 1),
            random_state=23,
        )
        desc_train_svd = svd_desc.fit_transform(desc_train)
        desc_test_svd  = svd_desc.transform(desc_test)
        print(f"   - desc SVD: {desc_train_svd.shape}, 설명력={svd_desc.explained_variance_ratio_.sum():.2%}")

        del desc_train, desc_test, vec_desc, svd_desc
        gc.collect()

        train_vec = np.hstack([name_train_svd, desc_train_svd]).astype(np.float32)
        test_vec  = np.hstack([name_test_svd,  desc_test_svd]).astype(np.float32)

        del name_train_svd, name_test_svd, desc_train_svd, desc_test_svd
        gc.collect()

        self.train_vectorized = pd.DataFrame(
            train_vec,
            columns=[f"name_{i}" for i in range(n_components)]
                  + [f"desc_{i}" for i in range(n_components)],
        )
        self.test_vectorized = pd.DataFrame(
            test_vec,
            columns=[f"name_{i}" for i in range(n_components)]
                  + [f"desc_{i}" for i in range(n_components)],
        )
        self._add_categorical_numeric_features()
        print(f"✅ TF-IDF 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}")
    # eof -----------------------------------

    # vectorize_text_fasttext ###############
    def vectorize_text_fasttext(
        self,
        text_columns=["name", "item_description"],
        fasttext_size=100,
        fasttext_window=5,
        fasttext_min_count=2,
        n_components=None,
    ):
        """FastText 벡터화 (AutoGluon/Tabular에 먹이기 좋은 형태)"""
        print("🔍 FastText 벡터화 시작...")

        # 정규화 텍스트 컬럼 생성
        for col in text_columns:
            raw = self.train[col].fillna("").astype(str)
            self.train[f"{col}_norm"] = raw.apply(self._simple_normalize)

            raw_t = self.test[col].fillna("").astype(str)
            self.test[f"{col}_norm"] = raw_t.apply(self._simple_normalize)

        # FastText 학습용 문장 리스트
        sentences = []
        for col in text_columns:
            norm_col = f"{col}_norm"
            sentences += [
                str(x).split()
                for x in pd.concat([self.train[norm_col], self.test[norm_col]], axis=0)
            ]

        print("   - FastText 학습 중...")
        ft_model = FastText(
            sentences,
            vector_size=fasttext_size,
            window=fasttext_window,
            min_count=fasttext_min_count,
            sg=1,
            workers=4,
        )

        def get_vector_from_norm(norm_text):
            words = str(norm_text).split()
            vectors = [ft_model.wv[w] for w in words if w in ft_model.wv]
            if not vectors:
                return np.zeros(fasttext_size, dtype=np.float32)
            return np.mean(vectors, axis=0).astype(np.float32)

        train_features = []
        test_features  = []

        for col in tqdm(text_columns, desc="FastText 벡터 생성"):
            norm_col = f"{col}_norm"
            train_features.append(
                np.vstack(self.train[norm_col].apply(get_vector_from_norm))
            )
            test_features.append(
                np.vstack(self.test[norm_col].apply(get_vector_from_norm))
            )

        train_vec = np.hstack(train_features).astype(np.float32)
        test_vec  = np.hstack(test_features).astype(np.float32)

        # 옵션: PCA로 차원 축소
        if n_components is not None and n_components < train_vec.shape[1]:
            from sklearn.decomposition import PCA
            print(f"   - PCA로 {train_vec.shape[1]} → {n_components} 축소")
            pca = PCA(n_components=n_components, random_state=42)
            train_vec = pca.fit_transform(train_vec)
            test_vec  = pca.transform(test_vec)

            dim_per_col = n_components // len(text_columns)
            col_names = [
                f"{col}_ft_{i}"
                for col in text_columns
                for i in range(dim_per_col)
            ]
        else:
            dim_per_col = fasttext_size
            col_names = [
                f"{col}_ft_{i}"
                for col in text_columns
                for i in range(dim_per_col)
            ]

        self.train_vectorized = pd.DataFrame(train_vec, columns=col_names)
        self.test_vectorized  = pd.DataFrame(test_vec,  columns=col_names)
        self._add_categorical_numeric_features()
        print(f"✅ FastText 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}")
    # eof -----------------------------------

    # vectorize_text_bert ###################
    def vectorize_text_bert(
        self,
        text_columns=["name", "item_description"],
        bert_model_name="all-MiniLM-L6-v2",
    ):
        """BERT 벡터화 (옵션)"""
        print(f"🔍 BERT 시작 (모델={bert_model_name})...")
        bert_model = SentenceTransformer(bert_model_name)
        train_features, test_features = [], []

        for col in text_columns:
            self.train[col] = self.train[col].fillna("").astype(str)
            self.test[col]  = self.test[col].fillna("").astype(str)

            print(f"   - {col} 인코딩 중...")
            train_features.append(
                bert_model.encode(
                    self.train[col].tolist(),
                    show_progress_bar=True,
                    batch_size=32,
                )
            )
            test_features.append(
                bert_model.encode(
                    self.test[col].tolist(),
                    show_progress_bar=True,
                    batch_size=32,
                )
            )

        train_vec = np.hstack(train_features).astype(np.float32)
        test_vec  = np.hstack(test_features).astype(np.float32)
        emb_dim   = bert_model.get_sentence_embedding_dimension()

        self.train_vectorized = pd.DataFrame(
            train_vec,
            columns=[f"{col}_bert_{i}" for col in text_columns for i in range(emb_dim)],
        )
        self.test_vectorized = pd.DataFrame(
            test_vec,
            columns=[f"{col}_bert_{i}" for col in text_columns for i in range(emb_dim)],
        )
        self._add_categorical_numeric_features()
        print(f"✅ BERT 완료: train {self.train_vectorized.shape}, test {self.test_vectorized.shape}")
    # eof -----------------------------------

    # _add_categorical_numeric_features #####
    def _add_categorical_numeric_features(self):
        """범주형/수치형 피처 추가"""
        categorical_features = [
            "main_cat", "sub_cat", "sub_sub_cat",
            "brand_name", "item_condition_id", "shipping",
        ]
        numeric_features = [
            "name_len_char", "name_len_word",
            "desc_len_char", "desc_len_word",
            "has_brand_in_name", "has_brand_in_desc",
        ]
        print("📌 범주형/수치형 피처 추가 중...")
        for col in categorical_features + numeric_features:
            if col in self.train.columns:
                self.train_vectorized[col] = self.train[col].reset_index(drop=True)
                self.test_vectorized[col]  = self.test[col].reset_index(drop=True)
        print(
            f"   - 범주형:{len(categorical_features)}개, "
            f"수치형:{len(numeric_features)}개, "
            f"총:{self.train_vectorized.shape[1]}개"
        )
    # eof -----------------------------------

    # vectorize_text ########################
    def vectorize_text(self, method="tfidf", **kwargs):
        """텍스트 벡터화 통합 인터페이스, method별 arg 제한"""
        if self.load_vectorized(method):
            return

        if method == "tfidf":
            allowed = {"max_features_name", "max_features_desc", "n_components"}
            args = {k: v for k, v in kwargs.items() if k in allowed}
            self.vectorize_text_tfidf(**args)

        elif method == "fasttext":
            allowed = {
                "text_columns", "fasttext_size",
                "fasttext_window", "fasttext_min_count",
                "n_components",
            }
            args = {k: v for k, v in kwargs.items() if k in allowed}
            self.vectorize_text_fasttext(**args)

        elif method == "bert":
            allowed = {"text_columns", "bert_model_name"}
            args = {k: v for k, v in kwargs.items() if k in allowed}
            self.vectorize_text_bert(**args)

        else:
            raise ValueError("method must be one of ['tfidf','fasttext','bert']")

        self.save_vectorized(method)
    # eof -----------------------------------

    # =======================
    # PyCaret 관련 메서드 (기존)
    # =======================

    def setup_pycaret(self, session_id=23, fold=3, use_gpu=False):
        """PyCaret 환경 설정 (AutoGluon과 별개, 필요 시 사용)"""
        print("🔧 PyCaret setup 시작...")
        categorical_cols = [
            "main_cat", "sub_cat", "sub_sub_cat",
            "brand_name", "item_condition_id", "shipping",
        ]
        existing_categorical = [
            col for col in categorical_cols if col in self.train_vectorized.columns
        ]
        print(f"   - 범주형:{len(existing_categorical)}개, 전체:{self.train_vectorized.shape[1]}개")
        self.setup_result = setup(
            data=self.train_vectorized.assign(
                price=self.train["price"].reset_index(drop=True)
            ),
            target="price",
            session_id=session_id,
            categorical_features=existing_categorical if existing_categorical else None,
            normalize=True,
            transformation=False,
            fold_strategy="kfold",
            fold=fold,
            use_gpu=use_gpu,
            n_jobs=4,
            verbose=True,
            html=False,
        )
        gc.collect()
        print("✅ PyCaret setup 완료")
    # eof -----------------------------------

    def find_and_blend_models(self, top_n=3, sort_metric="R2", use_kaggle_winners=True):
        """PyCaret: 모델 탐색 및 블렌딩"""
        if not self.setup_result:
            raise ValueError("먼저 setup_pycaret()를 실행하세요.")

        if use_kaggle_winners:
            print("🏆 Kaggle 상위권 5개 모델 학습...")
            top_models = [
                create_model(name, verbose=False)
                for name in tqdm(["lightgbm", "ridge", "catboost", "xgboost", "et"],
                                 desc="모델 학습")
            ]
            print("✅ 5개 모델 학습 완료")
        else:
            print(f"🔍 전체 모델 탐색 (상위 {top_n}개)...")
            top_models = compare_models(
                n_select=top_n, sort=sort_metric, turbo=True, verbose=True
            )
            if not isinstance(top_models, list):
                top_models = [top_models]

        print("🎯 선정 모델:")
        for i, m in enumerate(top_models, 1):
            print(f"   {i}. {str(m).split('(')[0]}")

        print(f"\n🔀 {len(top_models)}개 모델 블렌딩...")
        blended = blend_models(
            estimator_list=top_models,
            optimize=sort_metric,
            choose_better=True,
            verbose=True,
        )
        self.best_model = blended
        gc.collect()
        print(f"🏆 Blended model 완료 (기준={sort_metric})")

        # 버전/최신 모델 저장
        model_tag = f"blended_{sort_metric}"
        self.save_best_model(model_name=model_tag)

        return self.best_model
    # eof -----------------------------------

    def tune_best_model(self, n_iter=50, optimize_metric="R2"):
        """PyCaret: best_model 튜닝"""
        if not self.best_model:
            raise ValueError("먼저 find_and_blend_models()를 실행하세요.")
        print(f"⚙️ 튜닝 시작 (n_iter={n_iter})...")
        tuned = tune_model(
            self.best_model,
            optimize=optimize_metric,
            n_iter=n_iter,
            search_library="optuna",
            search_algorithm="tpe",
        )
        self.best_model = tuned
        print("✅ 튜닝 완료!")

        model_tag = f"tuned_{optimize_metric}"
        self.save_best_model(model_name=model_tag)
        return tuned
    # eof -----------------------------------

    def save_metrics(self, model_name=None):
        """PyCaret: best_model 성능 지표 저장"""
        if not self.best_model:
            raise ValueError("모델이 없습니다.")
        print("📊 성능 평가 중...")

        pred_df = predict_model(self.best_model, data=self.train_vectorized.copy())
        y_true  = np.expm1(self.train["price"].values)
        y_pred  = np.expm1(pred_df["prediction_label"].values)

        self.metrics = {
            "R2":   round(r2_score(y_true, y_pred), 4),
            "RMSE": round(mean_squared_error(y_true, y_pred, squared=False), 4),
            "MAE":  round(mean_absolute_error(y_true, y_pred), 4),
        }

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        model_name = model_name or str(self.best_model).split("(")[0]
        file_path = os.path.join(
            self.results_dir, f"{model_name}_metrics_{timestamp}.json"
        )

        with open(file_path, "w") as f:
            json.dump(self.metrics, f, indent=4)

        print(f"💾 Metrics 저장: {file_path}")
        print(
            f"   - R²={self.metrics['R2']}, "
            f"RMSE=${self.metrics['RMSE']:.2f}, "
            f"MAE=${self.metrics['MAE']:.2f}"
        )
    # eof -----------------------------------

    def predict_test(self, submission_file="submission.csv"):
        """PyCaret: best_model로 테스트 예측"""
        if not self.best_model:
            raise ValueError("먼저 find_and_blend_models()를 실행하세요.")
        print("📦 Test 예측 시작...")

        predictions = predict_model(self.best_model, data=self.test_vectorized.copy())
        price_pred  = np.expm1(predictions["prediction_label"].values)

        submission = pd.DataFrame(
            {"test_id": self.test["test_id"], "price": price_pred}
        )

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        path = os.path.join(self.results_dir, f"{timestamp}_{submission_file}")
        submission.to_csv(path, index=False)

        print(f"💾 Submission 저장: {path}")
        print(
            f"   - 가격 범위: ${price_pred.min():.2f} ~ "
            f"${price_pred.max():.2f}, 평균: ${price_pred.mean():.2f}"
        )
        return submission
    # eof -----------------------------------

    # save_vectorized #######################
    def save_vectorized(self, method="tfidf"):
        """벡터화 결과 저장"""
        train_path = os.path.join(self.model_dir, f"vectorized_{method}_train.pkl")
        test_path  = os.path.join(self.model_dir, f"vectorized_{method}_test.pkl")
        self.train_vectorized.to_pickle(train_path)
        self.test_vectorized.to_pickle(test_path)
        print(f"💾 {method} 벡터화 결과 저장 완료")
    # eof -----------------------------------

    # load_vectorized #######################
    def load_vectorized(self, method="tfidf"):
        """벡터화 결과 로드"""
        train_path = os.path.join(self.model_dir, f"vectorized_{method}_train.pkl")
        test_path  = os.path.join(self.model_dir, f"vectorized_{method}_test.pkl")
        if os.path.exists(train_path) and os.path.exists(test_path):
            self.train_vectorized = pd.read_pickle(train_path)
            self.test_vectorized  = pd.read_pickle(test_path)
            print(
                f"📂 {method} 벡터화 결과 로드: "
                f"train {self.train_vectorized.shape}, "
                f"test {self.test_vectorized.shape}"
            )
            return True
        print(f"⚠️ {method} 벡터화 결과 없음")
        return False
    # eof -----------------------------------

    # save_best_model ########################
    def save_best_model(self, model_name=None):
        """PyCaret best_model을 ../models에 저장 (버전 + 최신)"""
        if not self.best_model:
            raise ValueError("저장할 모델이 없습니다. 먼저 find_and_blend_models() 실행 필요.")

        os.makedirs("../models", exist_ok=True)

        base_name = model_name or str(self.best_model).split("(")[0]
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        versioned_path = os.path.join("../models", f"{base_name}_{timestamp}")
        save_model(self.best_model, versioned_path)

        latest_path   = os.path.join("../models", f"{base_name}_latest")
        save_model(self.best_model, latest_path)

        print(f"💾 모델 저장 완료: {versioned_path}.pkl (버전), {latest_path}.pkl (최신)")
    # eof -----------------------------------

    def load_saved_model(self, model_name, latest=True):
        """../models에서 PyCaret 모델 불러오기"""
        suffix = "_latest" if latest else ""
        path   = os.path.join("../models", f"{model_name}{suffix}")
        self.best_model = load_model(path)
        print(f"📂 모델 불러오기 완료: {path}.pkl")
        return self.best_model
    # eof -----------------------------------

    # =======================
    # 🔹 AutoGluon 관련 메서드
    # =======================

    def fit_autogluon(
        self,
        label_col="price",
        time_limit=3600,
        presets="good_quality",
        num_gpus=1,
        num_cpus="auto",
    ):
        """
        AutoGluon TabularPredictor 학습
        - label = log1p(price)
        - eval_metric = 'root_mean_squared_error' (RMSLE와 동일 개념)
        """
        if not hasattr(self, "train_vectorized"):
            raise ValueError("먼저 vectorize_text()를 실행하세요.")

        print("🚀 AutoGluon Tabular 학습 시작...")

        train_df = self.train_vectorized.copy()
        train_df[label_col] = self.train[label_col].reset_index(drop=True)

        self.ag_predictor = TabularPredictor(
            label=label_col,
            eval_metric="root_mean_squared_error",
            problem_type="regression",
        ).fit(
            train_data=train_df,
            time_limit=time_limit,
            presets=presets,
            num_gpus=num_gpus,   # GPU 1개 사용
            num_cpus=num_cpus,
        )

        print("✅ AutoGluon 학습 완료")
        print(self.ag_predictor.leaderboard(silent=True))
        return self.ag_predictor
    # eof -----------------------------------

    def evaluate_autogluon_on_train(
        self,
        save_json=True,
        model_name="AutoGluon_FastText",
    ):
        """
        AutoGluon predictor를 train 전체에서 평가
        - log-price → 실제 price 변환 후 R2/RMSE/MAE 계산
        """
        if self.ag_predictor is None:
            raise ValueError("먼저 fit_autogluon()을 실행하세요.")

        print("📊 AutoGluon train 성능 평가 중...")

        y_log_true = self.train["price"].values
        y_log_pred = self.ag_predictor.predict(self.train_vectorized).values

        y_true = np.expm1(y_log_true)
        y_pred = np.expm1(y_log_pred)

        r2   = r2_score(y_true, y_pred)
        rmse = mean_squared_error(y_true, y_pred, squared=False)
        mae  = mean_absolute_error(y_true, y_pred)

        self.metrics = {
            "R2":   round(r2, 4),
            "RMSE": round(rmse, 4),
            "MAE":  round(mae, 4),
        }

        print(
            f"   - R2={self.metrics['R2']:.4f}, "
            f"RMSE={self.metrics['RMSE']:.4f}, "
            f"MAE={self.metrics['MAE']:.4f}"
        )

        if save_json:
            timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
            file_path = os.path.join(
                self.results_dir,
                f"{model_name}_metrics_{timestamp}.json",
            )
            with open(file_path, "w") as f:
                json.dump(self.metrics, f, indent=4)
            print(f"💾 Metrics 저장: {file_path}")

        return self.metrics
    # eof -----------------------------------

    def predict_test_autogluon(self, submission_file="submission_ag_fasttext.csv"):
        """
        AutoGluon predictor로 test 예측 후 submission 생성
        - 예측값: log-price → expm1 → price
        """
        if self.ag_predictor is None:
            raise ValueError("먼저 fit_autogluon()을 실행하세요.")

        print("📦 AutoGluon Test 예측 시작...")

        y_log_pred = self.ag_predictor.predict(self.test_vectorized).values
        price_pred = np.expm1(y_log_pred)

        submission = pd.DataFrame(
            {"test_id": self.test["test_id"], "price": price_pred}
        )

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        path = os.path.join(self.results_dir, f"{timestamp}_{submission_file}")
        submission.to_csv(path, index=False)

        print(f"💾 Submission 저장: {path}")
        print(
            f"   - price 범위: {price_pred.min():.2f} ~ "
            f"{price_pred.max():.2f}, 평균: {price_pred.mean():.2f}"
        )
        return submission
    # eof -----------------------------------


# End of class ###########################

True
NVIDIA GeForce GTX 1660 SUPER


In [3]:
print("=" * 60)
print("Mercari Price Suggestion - v8 (PyCaret v7 + AutoGluon + fastText)")
print("=" * 60)

analyzer = MercariPyCaretAnalyzer7()

Mercari Price Suggestion - v8 (PyCaret v7 + AutoGluon + fastText)


In [4]:
# 1) 데이터 로딩 (log1p(price) + 언더샘플링 + 카테고리/길이 피처)
analyzer.load_data(undersample_frac=0.35)

📂 데이터 로딩 시작...
원본: train (1482535, 8), test (693359, 7)
⚠️ Stratified undersampling 적용: train (518582, 8)
🔄 희귀값 통합 중...
📏 피처 생성 중...
✅ 데이터 로드 완료: train (518582, 16), test (693359, 15)


In [5]:
# 2) FastText 벡터화 (상위권 스타일 + 정규화 텍스트 사용)
analyzer.vectorize_text(
    method="fasttext",
    text_columns=["name", "item_description"],
    fasttext_size=100,
    fasttext_window=5,
    fasttext_min_count=2,
    n_components=None,   # 필요시 PCA 사용: 예) 150
)

⚠️ fasttext 벡터화 결과 없음
🔍 FastText 벡터화 시작...
   - FastText 학습 중...


FastText 벡터 생성: 100%|██████████| 2/2 [02:32<00:00, 76.16s/it]


📌 범주형/수치형 피처 추가 중...
   - 범주형:6개, 수치형:6개, 총:212개
✅ FastText 완료: train (518582, 212), test (693359, 212)
💾 fasttext 벡터화 결과 저장 완료


In [6]:
# 3) AutoGluon 학습 (GPU 1개, 속도/성능 타협 preset)
analyzer.fit_autogluon(
    label_col="price",
    time_limit=3600,          # 여유 없으면 1800 정도로 줄여도 됨
    presets="good_quality",
    num_gpus=1,
    num_cpus="auto",
)

🚀 AutoGluon Tabular 학습 시작...


No path specified. Models will be saved in: "AutogluonModels\ag-20251205_033908"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.19
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          12
Memory Avail:       18.31 GB / 31.91 GB (57.4%)
Disk Space Avail:   282.05 GB / 465.09 GB (60.6%)
Presets specified: ['good_quality']
Using hyperparameters preset: hyperparameters='light'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`

✅ AutoGluon 학습 완료
                       model  score_val              eval_metric  \
0        WeightedEnsemble_L3  -0.463148  root_mean_squared_error   
1          LightGBMXT_BAG_L2  -0.463207  root_mean_squared_error   
2            LightGBM_BAG_L2  -0.465784  root_mean_squared_error   
3          LightGBMXT_BAG_L1  -0.467233  root_mean_squared_error   
4        WeightedEnsemble_L2  -0.467233  root_mean_squared_error   
5            LightGBM_BAG_L1  -0.499101  root_mean_squared_error   
6   WeightedEnsemble_L3_FULL        NaN  root_mean_squared_error   
7   WeightedEnsemble_L2_FULL        NaN  root_mean_squared_error   
8       LightGBM_BAG_L2_FULL        NaN  root_mean_squared_error   
9       LightGBM_BAG_L1_FULL        NaN  root_mean_squared_error   
10    LightGBMXT_BAG_L2_FULL        NaN  root_mean_squared_error   
11    LightGBMXT_BAG_L1_FULL        NaN  root_mean_squared_error   

    pred_time_val     fit_time  pred_time_val_marginal  fit_time_marginal  \
0      417.246400  2

In [7]:
# 4) train 성능 평가 + 지표 저장
analyzer.evaluate_autogluon_on_train(
    save_json=True,
    model_name="AutoGluon_FastText",
)

📊 AutoGluon train 성능 평가 중...
   - R2=0.6031, RMSE=24.3617, MAE=8.8239
💾 Metrics 저장: ../results\AutoGluon_FastText_metrics_20251205_134652.json


{'R2': 0.6031, 'RMSE': 24.3617, 'MAE': 8.8239}

In [8]:
# 5) test 예측 + submission 생성
analyzer.predict_test_autogluon(
    submission_file="submission_ag_fasttext.csv",
)

print("\n✅ AutoGluon + fastText 파이프라인 완료!")

📦 AutoGluon Test 예측 시작...
💾 Submission 저장: ../results\20251205_134822_submission_ag_fasttext.csv
   - price 범위: 2.85 ~ 649.17, 평균: 23.15

✅ AutoGluon + fastText 파이프라인 완료!
